<div style="background:#0B1F4B;padding:22px 26px;border-radius:12px;border:3px solid #C9A227;font-family:'Times New Roman',serif">
<img src="https://raw.githubusercontent.com/drssnaiml1-cell/26UAMPC204-Artificial-Intelligence/main/assets/logo.png" alt="Sanjivani University" width="92" style="float:right;background:#FFFFFF;border:2px solid #C9A227;border-radius:8px;padding:3px;margin-left:14px">
<div style="color:#C9A227;font-size:14px;letter-spacing:2px;font-weight:bold">26UAMPC204 · ARTIFICIAL INTELLIGENCE · ASSIGNMENT 1 · UNIT I · CO204.1</div>
<div style="color:#FFFFFF;font-size:30px;font-weight:bold;margin-top:6px">Agents in the Wild</div>
<div style="color:#F6EBC8;font-size:15px;margin-top:8px">Due: End of Week 3 · Name: ____________ · Roll No: ________</div>
<div style="clear:both"></div></div>

## 🧠 Part A — Think out of the box

Answer each in 250–400 words in the markdown cells below.

1. **[L5]** If a self-driving car is perfectly rational for its performance measure but still hits a pedestrian in a sudden event, is the car 'to blame'? Argue from the definition of rationality, then propose how the performance measure itself should change.
2. **[L6]** Invent a new agent type that does not fit any of the five classical architectures (reflex, model, goal, utility, learning). Describe its internal structure with a diagram and give a real problem where it beats all five.
3. **[L4]** Your phone's keyboard autocomplete is an agent. Write its PEAS, classify its environment along all seven properties, and identify which property makes it hardest to design.
4. **[L5]** Design a 'Reverse Turing Test' in which a machine must decide whether it is talking to a human or another AI. What questions would it ask, and why might this become important by 2030?

**Answer A1.1:** _write here_

**Answer A1.2:** _write here_

**Answer A1.3:** _write here_

**Answer A1.4:** _write here_

## 💻 Part B — Open-ended coding

The starter code runs as-is and prints a baseline. Complete every `TODO`.

### B1.1 · Open-ended: a smarter vacuum world

<div style="border-left:6px solid #1F3A7A;background:#EEF2F8;padding:12px 18px;border-radius:6px;font-family:'Times New Roman',serif;color:#1A202C">
<b style="color:#1F3A7A">Task</b><br>Extend the vacuum world to an n×n grid where dirt re-appears randomly. Design the best agent you can and justify its performance measure. Compare at least three agent designs over 500 runs.</div>

In [1]:
import random
def run(agent, n=4, steps=200, p_dirt=0.05, seed=0):
    rng = random.Random(seed); dirt = {(r, c): rng.random() < 0.5 for r in range(n) for c in range(n)}
    pos, score, mem = (0, 0), 0, {}
    for _ in range(steps):
        action = agent(pos, dirt[pos], mem, n)
        if action == "Suck" and dirt[pos]: dirt[pos] = False; score += 10
        elif action in ("Up", "Down", "Left", "Right"):
            dr, dc = {"Up": (-1, 0), "Down": (1, 0), "Left": (0, -1), "Right": (0, 1)}[action]
            pos = (min(max(pos[0] + dr, 0), n - 1), min(max(pos[1] + dc, 0), n - 1)); score -= 1
        for k in dirt:                                   # dirt re-appears
            if rng.random() < p_dirt: dirt[k] = True
    return score

def random_agent(pos, dirty, mem, n):
    return "Suck" if dirty else random.choice(["Up", "Down", "Left", "Right"])

print("Random agent mean score:", sum(run(random_agent, seed=s) for s in range(50)) / 50)
# TODO 1: write a model-based agent that remembers when it last visited each cell.
# TODO 2: write a utility-based agent that moves towards the cell most likely to be dirty.
# TODO 3: plot the score distributions of all agents and explain the winner.

Random agent mean score: 439.1


In [2]:
# Your solution for B1.1


### B1.2 · Open-ended: formulate your own search problem

<div style="border-left:6px solid #1F3A7A;background:#EEF2F8;padding:12px 18px;border-radius:6px;font-family:'Times New Roman',serif;color:#1A202C">
<b style="color:#1F3A7A">Task</b><br>Pick a real problem from your daily life (bus route to campus, cooking a meal, packing a bag). Formulate it with the five components and solve a small instance with BFS. Report the state-space size.</div>

In [3]:
from collections import deque
# Example formulation: the classic Farmer-Wolf-Goat-Cabbage river crossing.
# State: tuple of sides (0 = left, 1 = right) for (farmer, wolf, goat, cabbage)
def safe(s):
    f, w, g, c = s
    return not ((w == g != f) or (g == c != f))
def successors(s):
    f = s[0]
    for i in range(4):                     # farmer crosses alone (i=0) or with item i
        if s[i] == f:
            t = list(s); t[0] = 1 - f
            if i: t[i] = 1 - f
            t = tuple(t)
            if safe(t): yield t
def bfs(start, goal):
    q, par = deque([start]), {start: None}
    while q:
        s = q.popleft()
        if s == goal:
            path = []
            while s: path.append(s); s = par[s]
            return path[::-1]
        for n in successors(s):
            if n not in par: par[n] = s; q.append(n)
path = bfs((0, 0, 0, 0), (1, 1, 1, 1))
print(f"Solved in {len(path) - 1} crossings:", path)
# TODO: replace this with YOUR problem. State clearly: states, initial state, actions,
#       transition model, goal test and path cost. Count reachable states.

Solved in 7 crossings: [(0, 0, 0, 0), (1, 0, 1, 0), (0, 0, 1, 0), (1, 1, 1, 0), (0, 1, 0, 0), (1, 1, 0, 1), (0, 1, 0, 1), (1, 1, 1, 1)]


In [4]:
# Your solution for B1.2


### B1.3 · Open-ended: performance-measure stress test

<div style="border-left:6px solid #1F3A7A;background:#EEF2F8;padding:12px 18px;border-radius:6px;font-family:'Times New Roman',serif;color:#1A202C">
<b style="color:#1F3A7A">Task</b><br>Write two performance measures for the same agent that lead to visibly different behaviour. Show the difference with a plot and explain which you would ship.</div>

In [5]:
# Starter idea: an elevator agent choosing which floor to serve next.
import random
def simulate(choose, steps=300, floors=10, seed=1):
    rng = random.Random(seed); waiting, pos, waits = {}, 0, []
    for t in range(steps):
        if rng.random() < 0.6: waiting.setdefault(rng.randrange(floors), []).append(t)
        if waiting:
            target = choose(pos, waiting)
            pos += (target > pos) - (target < pos)          # move one floor per time step
            if pos in waiting: waits += [t - a for a in waiting.pop(pos)]
    return sum(waits) / max(len(waits), 1), max(waits or [0])
nearest = lambda pos, w: min(w, key=lambda f: abs(f - pos))
oldest  = lambda pos, w: min(w, key=lambda f: min(w[f]))
print("nearest-first (mean wait, max wait):", simulate(nearest))
print("oldest-first  (mean wait, max wait):", simulate(oldest))
# TODO: define the performance measure you think is fair, design an agent for it,
#       and discuss the trade-off between mean and worst-case waiting time.

nearest-first (mean wait, max wait): (4.331460674157303, 20)
oldest-first  (mean wait, max wait): (4.936781609195402, 16)


In [6]:
# Your solution for B1.3


## 📋 Reflection

What surprised you? What would you do with one more week? How did you use AI tools, if at all?